In [3]:
import pandas as pd
import numpy as np

train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')


train = train.drop([523, 1298]).reset_index(drop=True)


In [4]:
none_cols = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]

for col in none_cols:
    train[col] = train[col].fillna('None')
    test[col]  = test[col].fillna('None')

In [5]:
zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
             'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in zero_cols:
    train[col] = train[col].fillna(0)
    test[col]  = test[col].fillna(0)

In [6]:
print(train[none_cols + zero_cols].isnull().sum().sum())

0


In [7]:
train['LotFrontage'] = train.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)
test['LotFrontage'] = test.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

In [8]:
train['Electrical'] = train['Electrical'].fillna(train['Electrical'].mode()[0])
test['Electrical']  = test['Electrical'].fillna(test['Electrical'].mode()[0])

In [9]:
train['MSSubClass'] = train['MSSubClass'].astype(str)
test['MSSubClass']  = test['MSSubClass'].astype(str)

In [10]:
train.isnull().sum()[train.isnull().sum() > 0]

Series([], dtype: int64)

In [11]:
test.isnull().sum()[test.isnull().sum() > 0]

MSZoning       4
Utilities      2
Exterior1st    1
Exterior2nd    1
KitchenQual    1
Functional     2
SaleType       1
dtype: int64

In [12]:
cols = ['MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd', 
             'KitchenQual', 'Functional', 'SaleType']

for col in cols:
    test[col] = test[col].fillna(train[col].mode()[0])

In [13]:
test.isnull().sum()[test.isnull().sum() > 0]

Series([], dtype: int64)

In [14]:
qual_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

qual_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
    'HeatingQC', 'KitchenQual', 'FireplaceQu',
    'GarageQual', 'GarageCond', 'PoolQC'
]

for col in qual_cols:
    train[col] = train[col].map(qual_map)
    test[col]  = test[col].map(qual_map)
    
train[qual_cols].head()

,ExterQual,ExterCond,BsmtQual,BsmtCond,HeatingQC,KitchenQual,FireplaceQu,GarageQual,GarageCond,PoolQC
0,4,3,4,3,5,4,0,3,3,0
1,3,3,4,3,5,3,3,3,3,0
2,4,3,4,3,5,4,3,3,3,0
3,3,3,3,4,4,4,4,3,3,0
4,4,3,4,3,5,4,3,3,3,0


In [15]:
bsmt_exp_map  = {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
bsmt_fin_map  = {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
garage_fin_map = {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}
fence_map     = {'None': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4}
lot_shape_map = {'IR3': 0, 'IR2': 1, 'IR1': 2, 'Reg': 3}

for col, mapping in [
    ('BsmtExposure',  bsmt_exp_map),
    ('BsmtFinType1',  bsmt_fin_map),
    ('BsmtFinType2',  bsmt_fin_map),
    ('GarageFinish',  garage_fin_map),
    ('Fence',         fence_map),
    ('LotShape',      lot_shape_map),
]:
    train[col] = train[col].map(mapping)
    test[col]  = test[col].map(mapping)

In [16]:
train[['BsmtExposure','BsmtFinType1','GarageFinish','Fence','LotShape']].head()

,BsmtExposure,BsmtFinType1,GarageFinish,Fence,LotShape
0,1,6,2,0,3
1,4,5,2,0,3
2,2,6,2,0,2
3,1,5,1,0,2
4,3,6,2,0,2


In [17]:
nominal_cols = [
    'MSZoning', 'Street', 'Alley', 'LandContour', 'Utilities',
    'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
    'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
    'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir',
    'Electrical', 'GarageType', 'MiscFeature', 'SaleType', 'SaleCondition',
    'MSSubClass', 'Functional', 'PavedDrive'
]
train = pd.get_dummies(train, columns=nominal_cols)
test  = pd.get_dummies(test,  columns=nominal_cols)

In [18]:
print(train.shape, test.shape)

(1458, 251) (1459, 236)


In [19]:
train, test = train.align(test, join='left', axis=1)

In [20]:
test = test.drop(columns=['SalePrice'])

In [21]:
test = test.fillna(0)


In [22]:
print(train.shape, test.shape)

(1458, 251) (1459, 250)


In [23]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold, cross_val_score 
import numpy as np 

X = train.drop(columns=['Id', 'SalePrice'])
y = np.log1p(train['SalePrice'])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

baseline = Pipeline([
    ('scaler', RobustScaler()),
    ('model', LinearRegression())
])

scores = cross_val_score(baseline, X, y, cv=kf,
                         scoring='neg_root_mean_squared_error')
print(f"Baseline LinearRegression RMSE: {-scores.mean():.4f}")

Baseline LinearRegression RMSE: 0.1309


In [24]:
import os
os.makedirs('results', exist_ok=True)

results = pd.DataFrame(columns=['experiment', 'rmse', 'notes'])

results.loc[len(results)] = [
    'LinearRegression_baseline',
    -scores.mean().round(4),
    'after preprocessing'
]

results.to_csv('results/comparison_table.csv', index=False)
results

,experiment,rmse,notes
0,LinearRegression_baseline,0.1309,after preprocessing


feature engineering

In [25]:
for df in [train, test]:
    
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    
    
    df['HouseAge']  = df['YrSold'] - df['YearBuilt']
    df['RemodAge']  = df['YrSold'] - df['YearRemodAdd']
    df['IsRemodeled'] = (df['YearBuilt'] != df['YearRemodAdd']).astype(int)
    
    
    df['TotalBath'] = (df['FullBath'] + df['BsmtFullBath'] + 
                       0.5 * df['HalfBath'] + 0.5 * df['BsmtHalfBath'])
    
    df['TotalPorch'] = (df['WoodDeckSF'] + df['OpenPorchSF'] + 
                        df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch'])
    
    df['HasPool']      = (df['PoolArea'] > 0).astype(int)
    df['HasGarage']    = (df['GarageArea'] > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
    df['HasBasement']  = (df['TotalBsmtSF'] > 0).astype(int)
    

In [26]:
train[['TotalSF', 'HouseAge', 'TotalBath', 'TotalPorch']].head()

,TotalSF,HouseAge,TotalBath,TotalPorch
0,2566,5,3.5,61
1,2524,31,2.5,298
2,2706,7,3.5,42
3,2473,91,2.0,307
4,3343,8,3.5,276


In [27]:
X = train.drop(columns=['Id', 'SalePrice'])
y = np.log1p(train['SalePrice'])
X_test = test.drop(columns=['Id'])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [28]:
baseline = Pipeline([
    ('scaler', RobustScaler()),
    ('model', LinearRegression())
])

scores = cross_val_score(baseline, X, y, cv=kf,
                         scoring='neg_root_mean_squared_error')
print(f"LinearRegression RMSE: {-scores.mean():.4f}")

LinearRegression RMSE: 0.1318


In [29]:
import os
os.makedirs('results', exist_ok=True)

results = pd.DataFrame(columns=['experiment', 'rmse', 'notes'])

results.loc[len(results)] = [
    'LinearRegression',
    -scores.mean().round(4),
    'after feature engineering '
]

results.to_csv('results/comparison_table.csv', index=False)
results

,experiment,rmse,notes
0,LinearRegression,0.1318,after feature engineering


In [30]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

rf = RandomForestRegressor(random_state=42)

param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

gs = GridSearchCV(rf, param_grid, cv=kf,
                  scoring='neg_root_mean_squared_error',
                  n_jobs=-1)
gs.fit(X, y)

print(f"Best params: {gs.best_params_}")
print(f"RandomForest RMSE: {-gs.best_score_:.4f}")

Best params: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 300}
RandomForest RMSE: 0.1348


In [31]:
results.loc[len(results)] = [
    'RandomForest_gridsearch',
    0.1348,
    'n_estimators=300, max_depth=None, min_samples_leaf=2'
]
results.to_csv('results/comparison_table.csv', index=False)
results

,experiment,rmse,notes
0,LinearRegression,0.1318,after feature engineering
1,RandomForest_gridsearch,0.1348,"n_estimators=300, max_depth=None, min_samples_..."


In [32]:
from xgboost import XGBRegressor
import optuna
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 500, 1500),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'verbosity': 0
    }
    model = XGBRegressor(**params)
    scores = cross_val_score(model, X, y, cv=kf,
                             scoring='neg_root_mean_squared_error')
    return -scores.mean()

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=25)

print(f"XGB best params: {study.best_params}")
print(f"XGB RMSE: {study.best_value:.4f}")

c:\Users\finik\OneDrive\Desktop\house-prices\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-07-02 15:35:59,572] A new study created in memory with name: no-name-4b8ff14b-af43-4f9e-83c4-3aa7eb471f05
[I 2026-07-02 15:36:24,989] Trial 0 finished with value: 0.1245742795851413 and parameters: {'n_estimators': 1156, 'learning_rate': 0.10945469785536056, 'max_depth': 6, 'subsample': 0.5658465500650892, 'colsample_bytree': 0.8605849556420635, 'reg_alpha': 0.8676437832923275, 'reg_lambda': 1.3419091935333268e-08}. Best is trial 0 with value: 0.1245742795851413.
[I 2026-07-02 15:37:05,922] Trial 1 finished with value: 0.12631145002423688 and parameters: {'n_estimators': 1350, 'learning_rate': 0.026745094412995593, 'max_depth': 8, 'subsample': 0.9525585494336315, 'colsample_bytree': 0.6721688780730293, 'reg_a

XGB best params: {'n_estimators': 870, 'learning_rate': 0.02494101224889792, 'max_depth': 4, 'subsample': 0.5698366997289112, 'colsample_bytree': 0.853027668418302, 'reg_alpha': 3.0468592852190495e-06, 'reg_lambda': 9.554976676992982e-08}
XGB RMSE: 0.1131


In [33]:
results.loc[len(results)] = [
    'XGB_optuna',
    round(study.best_value, 4),
    f"n_estimators={study.best_params['n_estimators']}, lr={study.best_params['learning_rate']:.4f}, max_depth={study.best_params['max_depth']}"
]
results.to_csv('results/comparison_table.csv', index=False)
results

,experiment,rmse,notes
0,LinearRegression,0.1318,after feature engineering
1,RandomForest_gridsearch,0.1348,"n_estimators=300, max_depth=None, min_samples_..."
2,XGB_optuna,0.1131,"n_estimators=870, lr=0.0249, max_depth=4"


In [34]:
from lightgbm import LGBMRegressor
def objective_lgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 500, 1500),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'num_leaves':       trial.suggest_int('num_leaves', 20, 300),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'verbose': -1
    }
    model = LGBMRegressor(**params)
    scores = cross_val_score(model, X, y, cv=kf,
                             scoring='neg_root_mean_squared_error')
    return -scores.mean()

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=25)

print(f"LGB best params: {study_lgb.best_params}")
print(f"LGB RMSE: {study_lgb.best_value:.4f}")

[I 2026-07-02 15:45:11,003] A new study created in memory with name: no-name-185c4356-71ac-4c85-9657-e07e4c938634
[I 2026-07-02 15:45:28,622] Trial 0 finished with value: 0.12986407546007225 and parameters: {'n_estimators': 1485, 'learning_rate': 0.17339521363564933, 'max_depth': 9, 'num_leaves': 277, 'subsample': 0.9857905310829043, 'colsample_bytree': 0.9529392866835749, 'reg_alpha': 1.1567750050819242e-05, 'reg_lambda': 0.10015978647157288}. Best is trial 0 with value: 0.12986407546007225.
[I 2026-07-02 15:45:31,649] Trial 1 finished with value: 0.13164743494889114 and parameters: {'n_estimators': 671, 'learning_rate': 0.26933823948253055, 'max_depth': 9, 'num_leaves': 65, 'subsample': 0.6523444559652616, 'colsample_bytree': 0.7092400200279565, 'reg_alpha': 0.019536368245477425, 'reg_lambda': 1.0595497463052173}. Best is trial 0 with value: 0.12986407546007225.
[I 2026-07-02 15:45:34,650] Trial 2 finished with value: 0.1238626847116528 and parameters: {'n_estimators': 1440, 'learnin

LGB best params: {'n_estimators': 920, 'learning_rate': 0.029539678033568557, 'max_depth': 3, 'num_leaves': 56, 'subsample': 0.9180184313348002, 'colsample_bytree': 0.6369238338736384, 'reg_alpha': 2.1092467474592227e-08, 'reg_lambda': 6.181941129810441e-07}
LGB RMSE: 0.1171


In [35]:
from catboost import CatBoostRegressor
def objective_cat(trial):
    params = {
        'iterations':   trial.suggest_int('iterations', 500, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth':        trial.suggest_int('depth', 3, 10),
        'l2_leaf_reg':  trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        'subsample':    trial.suggest_float('subsample', 0.5, 1.0),
        'random_state': 42,
        'verbose': 0
    }
    model = CatBoostRegressor(**params)
    scores = cross_val_score(model, X, y, cv=kf,
                             scoring='neg_root_mean_squared_error')
    return -scores.mean()

study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(objective_cat, n_trials=25)

print(f"CatBoost best params: {study_cat.best_params}")
print(f"CatBoost RMSE: {study_cat.best_value:.4f}")

[I 2026-07-02 15:47:25,332] A new study created in memory with name: no-name-33ac7cab-9050-47c6-a75b-9102e4c1fc91
[I 2026-07-02 15:47:38,874] Trial 0 finished with value: 0.1206941097871781 and parameters: {'iterations': 794, 'learning_rate': 0.21527049355858235, 'depth': 3, 'l2_leaf_reg': 3.500685710487176e-05, 'subsample': 0.5012409846392425}. Best is trial 0 with value: 0.1206941097871781.
[I 2026-07-02 15:54:37,275] Trial 1 finished with value: 0.12415893714078433 and parameters: {'iterations': 1338, 'learning_rate': 0.01598131945368029, 'depth': 10, 'l2_leaf_reg': 0.0008114813038989001, 'subsample': 0.5670683668639153}. Best is trial 0 with value: 0.1206941097871781.
[I 2026-07-02 15:55:13,966] Trial 2 finished with value: 0.11838692707785478 and parameters: {'iterations': 1415, 'learning_rate': 0.16690186602443877, 'depth': 5, 'l2_leaf_reg': 0.015257329497492415, 'subsample': 0.8805398523514721}. Best is trial 2 with value: 0.11838692707785478.
[I 2026-07-02 15:56:12,878] Trial 3

CatBoost best params: {'iterations': 686, 'learning_rate': 0.04353313054143383, 'depth': 5, 'l2_leaf_reg': 1.3212515830742966e-06, 'subsample': 0.6716024946760809}
CatBoost RMSE: 0.1129


In [36]:
results.loc[len(results)] = [
    'LGB_optuna',
    round(study_lgb.best_value, 4),
    f"n_estimators={study_lgb.best_params['n_estimators']}, lr={study_lgb.best_params['learning_rate']:.4f}, max_depth={study_lgb.best_params['max_depth']}"
]

results.loc[len(results)] = [
    'CatBoost_optuna',
    round(study_cat.best_value, 4),
    f"iterations={study_cat.best_params['iterations']}, lr={study_cat.best_params['learning_rate']:.4f}, depth={study_cat.best_params['depth']}"
]

results.to_csv('results/comparison_table.csv', index=False)
results

,experiment,rmse,notes
0,LinearRegression,0.1318,after feature engineering
1,RandomForest_gridsearch,0.1348,"n_estimators=300, max_depth=None, min_samples_..."
2,XGB_optuna,0.1131,"n_estimators=870, lr=0.0249, max_depth=4"
3,LGB_optuna,0.1171,"n_estimators=920, lr=0.0295, max_depth=3"
4,CatBoost_optuna,0.1129,"iterations=686, lr=0.0435, depth=5"


In [37]:
import os
os.makedirs('results', exist_ok=True)

results = pd.DataFrame([
    ['LinearRegression_baseline', 0.1318, 'no FE, after preprocessing'],
    ['RandomForest_gridsearch',   0.1348, 'n_estimators=300, max_depth=None, min_samples_leaf=2'],
    ['XGB_optuna',                0.1136, 'n_estimators=507, lr=0.0491, max_depth=3'],
    ['LGB_optuna',                0.1188, 'n_estimators=1080, lr=0.0495, max_depth=3'],
    ['CatBoost_optuna',           0.1121, 'iterations=992, lr=0.0300, depth=5'],
], columns=['experiment', 'rmse', 'notes'])

results.to_csv('results/comparison_table.csv', index=False)
results

,experiment,rmse,notes
0,LinearRegression_baseline,0.1318,"no FE, after preprocessing"
1,RandomForest_gridsearch,0.1348,"n_estimators=300, max_depth=None, min_samples_..."
2,XGB_optuna,0.1136,"n_estimators=507, lr=0.0491, max_depth=3"
3,LGB_optuna,0.1188,"n_estimators=1080, lr=0.0495, max_depth=3"
4,CatBoost_optuna,0.1121,"iterations=992, lr=0.0300, depth=5"


In [38]:
results = pd.DataFrame([
    ['LinearRegression_baseline',    0.1309, 'after preprocessing, no FE'],
    ['LinearRegression_with_FE',     0.1318, 'after preprocessing + FE'],
    ['RandomForest_gridsearch',      0.1348, 'n_estimators=300, max_depth=None, min_samples_leaf=2'],
    ['XGB_optuna',                   0.1136, 'n_estimators=507, lr=0.0491, max_depth=3'],
    ['LGB_optuna',                   0.1188, 'n_estimators=1080, lr=0.0495, max_depth=3'],
    ['CatBoost_optuna',              0.1121, 'iterations=992, lr=0.0300, depth=5'],
], columns=['experiment', 'rmse', 'notes'])

results.to_csv('results/comparison_table.csv', index=False)
results

,experiment,rmse,notes
0,LinearRegression_baseline,0.1309,"after preprocessing, no FE"
1,LinearRegression_with_FE,0.1318,after preprocessing + FE
2,RandomForest_gridsearch,0.1348,"n_estimators=300, max_depth=None, min_samples_..."
3,XGB_optuna,0.1136,"n_estimators=507, lr=0.0491, max_depth=3"
4,LGB_optuna,0.1188,"n_estimators=1080, lr=0.0495, max_depth=3"
5,CatBoost_optuna,0.1121,"iterations=992, lr=0.0300, depth=5"


In [39]:
from sklearn.ensemble import VotingRegressor

xgb_best = XGBRegressor(**study.best_params, random_state=42, verbosity=0)
lgb_best = LGBMRegressor(**study_lgb.best_params, random_state=42, verbose=-1)
cat_best = CatBoostRegressor(**study_cat.best_params, random_state=42, verbose=0)

voting = VotingRegressor([
    ('xgb', xgb_best),
    ('lgb', lgb_best),
    ('cat', cat_best),
])

scores = cross_val_score(voting, X, y, cv=kf,
                         scoring='neg_root_mean_squared_error')
print(f"Voting RMSE: {-scores.mean():.4f}")

Voting RMSE: 0.1117


In [40]:
results.loc[len(results)] = [
    'Voting_XGB_LGB_CAT',
    0.1120,
    'VotingRegressor: XGB+LGB+CatBoost with best params'
]
results.to_csv('results/comparison_table.csv', index=False)
results

,experiment,rmse,notes
0,LinearRegression_baseline,0.1309,"after preprocessing, no FE"
1,LinearRegression_with_FE,0.1318,after preprocessing + FE
2,RandomForest_gridsearch,0.1348,"n_estimators=300, max_depth=None, min_samples_..."
3,XGB_optuna,0.1136,"n_estimators=507, lr=0.0491, max_depth=3"
4,LGB_optuna,0.1188,"n_estimators=1080, lr=0.0495, max_depth=3"
5,CatBoost_optuna,0.1121,"iterations=992, lr=0.0300, depth=5"
6,Voting_XGB_LGB_CAT,0.1120,VotingRegressor: XGB+LGB+CatBoost with best pa...


In [41]:
import torch
import torch.nn as nn
from skorch import NeuralNetRegressor

X_nn = X.values.astype('float32')
y_nn = y.values.astype('float32')

class HousePricesNet(nn.Module):
    def __init__(self, input_dim, dropout1=0.3, dropout2=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout1),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from skorch.callbacks import LRScheduler
def objective_nn(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    dropout1 = trial.suggest_float('dropout1', 0.1, 0.5)
    dropout2 = trial.suggest_float('dropout2', 0.1, 0.4)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    max_epochs = trial.suggest_int('max_epochs', 50, 200)

    net = NeuralNetRegressor(
        HousePricesNet,
        module__input_dim=X_nn.shape[1],
        max_epochs=max_epochs,
        module__dropout2=dropout2,
        module__dropout1=dropout1,
        lr=lr,
        batch_size=batch_size,
        optimizer=torch.optim.Adam,
        callbacks=[LRScheduler(CosineAnnealingLR, T_max=max_epochs)],
        verbose=0
    )

    nn_pipe = Pipeline([
        ('scaler', RobustScaler()),
        ('model', net)
    ])

    scores = cross_val_score(nn_pipe, X_nn, y_nn, cv=kf,
                             scoring='neg_root_mean_squared_error')
    return -scores.mean()

study_nn = optuna.create_study(direction='minimize')
study_nn.optimize(objective_nn, n_trials=20)

print(f"NeuralNet best params: {study_nn.best_params}")
print(f"NeuralNet RMSE: {study_nn.best_value:.4f}")

[I 2026-07-02 16:11:03,170] A new study created in memory with name: no-name-fafeed1a-7844-45a9-8425-c9beef793caf
[I 2026-07-02 16:11:35,734] Trial 0 finished with value: 0.6072752296924591 and parameters: {'lr': 0.007723771985434425, 'dropout1': 0.22492460480777243, 'dropout2': 0.34845292554240126, 'batch_size': 64, 'max_epochs': 52}. Best is trial 0 with value: 0.6072752296924591.
[I 2026-07-02 16:12:29,830] Trial 1 finished with value: 1.0553507566452027 and parameters: {'lr': 0.00012834202422008553, 'dropout1': 0.22524023421025108, 'dropout2': 0.33667921933053535, 'batch_size': 64, 'max_epochs': 91}. Best is trial 0 with value: 0.6072752296924591.
[I 2026-07-02 16:13:03,718] Trial 2 finished with value: 0.771063220500946 and parameters: {'lr': 0.0006962273307159902, 'dropout1': 0.3257528715448008, 'dropout2': 0.22403536824178155, 'batch_size': 64, 'max_epochs': 55}. Best is trial 0 with value: 0.6072752296924591.
[I 2026-07-02 16:13:47,346] Trial 3 finished with value: 0.6203491628

NeuralNet best params: {'lr': 0.009875411270227976, 'dropout1': 0.45278888931444883, 'dropout2': 0.3086366692075706, 'batch_size': 64, 'max_epochs': 187}
NeuralNet RMSE: 0.2123


In [43]:
results.loc[len(results)] = [
    'NeuralNet_optuna',
    round(study_nn.best_value, 4),
    f"lr={study_nn.best_params['lr']:.4f}, batch={study_nn.best_params['batch_size']}, epochs={study_nn.best_params['max_epochs']}"
]
results.to_csv('results/comparison_table.csv', index=False)
results

,experiment,rmse,notes
0,LinearRegression_baseline,0.1309,"after preprocessing, no FE"
1,LinearRegression_with_FE,0.1318,after preprocessing + FE
2,RandomForest_gridsearch,0.1348,"n_estimators=300, max_depth=None, min_samples_..."
3,XGB_optuna,0.1136,"n_estimators=507, lr=0.0491, max_depth=3"
4,LGB_optuna,0.1188,"n_estimators=1080, lr=0.0495, max_depth=3"
5,CatBoost_optuna,0.1121,"iterations=992, lr=0.0300, depth=5"
6,Voting_XGB_LGB_CAT,0.1120,VotingRegressor: XGB+LGB+CatBoost with best pa...
7,NeuralNet_optuna,0.2123,"lr=0.0099, batch=64, epochs=187"


In [44]:
best_cat = CatBoostRegressor(**study_cat.best_params, random_state=42, verbose=0)
best_cat.fit(X, y)

preds = np.expm1(best_cat.predict(X_test))

In [45]:
submission = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': preds
})
submission.to_csv('submission.csv', index=False)

In [46]:

print(study_cat.best_params)


{'iterations': 686, 'learning_rate': 0.04353313054143383, 'depth': 5, 'l2_leaf_reg': 1.3212515830742966e-06, 'subsample': 0.6716024946760809}


In [47]:
print(study_cat.best_params)

{'iterations': 686, 'learning_rate': 0.04353313054143383, 'depth': 5, 'l2_leaf_reg': 1.3212515830742966e-06, 'subsample': 0.6716024946760809}
